# bcrypt

Smoke test for [bcrypt](https://pypi.org/project/bcrypt/): password hashing with `hashpw` and
`checkpw`, salts and cost factors, published known-answer vectors, the `bcrypt_pbkdf` key
derivation function (`kdf`), and input validation. Passwords are generated at run time or are
short dummy values.

## Example 1: Hash and verify a password

In [ ]:
import secrets

import bcrypt

print(f"bcrypt {getattr(bcrypt, '__version__', 'unknown')}")

# A throwaway password generated at run time; bcrypt works on bytes, not text.
pw = secrets.token_urlsafe(16).encode("ascii")

salt = bcrypt.gensalt()
hashed = bcrypt.hashpw(pw, salt)
print(f"salt: {salt.decode()}")
print(f"hash: {hashed[:29].decode()}... ({len(hashed)} bytes)")

# Modular crypt format: $2b$<cost>$<22-character salt><31-character checksum>.
# gensalt() defaults to the 2b prefix and a cost factor of 12.
assert hashed.startswith(b"$2b$12$"), hashed[:7]
assert len(hashed) == 60, len(hashed)
assert hashed[:29] == salt, "the salt is stored at the start of the hash"

# The stored hash can be passed back as the salt: re-hashing gives the same value.
assert bcrypt.hashpw(pw, hashed) == hashed
assert bcrypt.checkpw(pw, hashed), "checkpw rejected the password the hash was made from"
assert not bcrypt.checkpw(pw + b"x", hashed), "checkpw accepted a different password"
assert not bcrypt.checkpw(b"wrong", hashed), "checkpw accepted a wrong password"
print("hashpw / checkpw round trip OK")

## Example 2: Unique salts, cost factors and the 2a prefix

In [ ]:
import secrets
import time

import bcrypt

pw = secrets.token_urlsafe(12).encode("ascii")

# Every gensalt() call returns a new random salt, so the same password hashes differently.
first = bcrypt.hashpw(pw, bcrypt.gensalt(rounds=4))
second = bcrypt.hashpw(pw, bcrypt.gensalt(rounds=4))
assert first != second, "two hashes of the same password share a salt"
assert bcrypt.checkpw(pw, first) and bcrypt.checkpw(pw, second)
print("same password, different salts -> different hashes, both verify")

# The cost factor is log2 of the number of key-expansion rounds and is recorded in the hash.
for cost in (4, 5, 6, 8):
    start = time.perf_counter()
    hashed = bcrypt.hashpw(pw, bcrypt.gensalt(rounds=cost))
    elapsed = time.perf_counter() - start
    assert hashed.startswith(b"$2b$%02d$" % cost), hashed[:7]
    assert bcrypt.checkpw(pw, hashed)
    print(f"cost {cost:2d}: {elapsed * 1000:7.1f} ms  {hashed[:7].decode()}")

# Hashes with the older 2a prefix are produced and verified too.
legacy = bcrypt.hashpw(pw, bcrypt.gensalt(rounds=4, prefix=b"2a"))
assert legacy.startswith(b"$2a$04$"), legacy[:7]
assert bcrypt.checkpw(pw, legacy)
print("2a prefix OK")

## Example 3: Known-answer vectors for hashpw and kdf

In [ ]:
import warnings

import bcrypt

# Published bcrypt test vectors (crypt_blowfish / OpenBSD): fixed salt in, fixed hash out.
VECTORS = [
    (b"", b"$2a$05$CCCCCCCCCCCCCCCCCCCCC.7uG0VCzI2bS7j6ymqJi9CdcdxiRTWNy"),
    (b"U*U", b"$2a$05$CCCCCCCCCCCCCCCCCCCCC.E5YPO9kmyuRGyh0XouQYb4YMJKvyOeW"),
    (b"U*U*", b"$2a$05$CCCCCCCCCCCCCCCCCCCCC.VGOzA784oUp/Z0DY336zx7pLYAy0lwK"),
    (b"U*U*U", b"$2a$05$XXXXXXXXXXXXXXXXXXXXXOAcXxm9kjPGEMsLznoKqmqw7tc8WCx4a"),
]
for plain, expected in VECTORS:
    assert bcrypt.hashpw(plain, expected[:29]) == expected, (plain, expected)
    assert bcrypt.checkpw(plain, expected), plain
print(f"{len(VECTORS)} hashpw vectors match")

# bcrypt_pbkdf: derive key material from a passphrase, e.g. for encrypting a private key file.
with warnings.catch_warnings():
    # kdf() warns below 50 rounds; 4 rounds keeps the test fast.
    warnings.simplefilter("ignore")
    derived = bcrypt.kdf(password=b"password", salt=b"salt", desired_key_bytes=32, rounds=4)
# Known-answer vector from the bcrypt project's kdf tests (its first rounds=4 case).
assert derived.hex() == "5bbf0cc293587f1c3635555c27796598d47e579071bf427e9d8fbe842aba34d9", derived.hex()
print(f"kdf vector matches: {derived.hex()[:16]}...")

# The derivation is deterministic and returns exactly the number of bytes asked for.
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    again = bcrypt.kdf(password=b"password", salt=b"salt", desired_key_bytes=32, rounds=4)
    longer = bcrypt.kdf(password=b"password", salt=b"salt", desired_key_bytes=64, rounds=4)
    other_salt = bcrypt.kdf(password=b"password", salt=b"pepper", desired_key_bytes=32, rounds=4)
assert again == derived
assert len(longer) == 64
assert other_salt != derived
print("kdf is deterministic and salt-dependent")

## Example 4: Input validation

In [ ]:
import bcrypt

salt = bcrypt.gensalt(rounds=4)

# Text must be encoded to bytes before hashing.
try:
    bcrypt.hashpw("text", salt)
except TypeError:
    print("str password rejected with TypeError")
else:
    raise AssertionError("hashpw accepted a str password")

# The cost factor must be between 4 and 31.
for bad_rounds in (3, 32):
    try:
        bcrypt.gensalt(rounds=bad_rounds)
    except ValueError:
        print(f"rounds={bad_rounds} rejected")
    else:
        raise AssertionError(f"gensalt accepted rounds={bad_rounds}")

# Only the 2a and 2b prefixes are generated.
try:
    bcrypt.gensalt(prefix=b"2x")
except ValueError:
    print("prefix 2x rejected")
else:
    raise AssertionError("gensalt accepted prefix 2x")

# A malformed salt or stored hash is an error, not a silent mismatch.
for bad in (b"$2b$04$short", b"not-a-bcrypt-hash"):
    try:
        bcrypt.hashpw(b"x", bad)
    except ValueError:
        print(f"malformed salt {bad!r} rejected")
    else:
        raise AssertionError(f"hashpw accepted the malformed salt {bad!r}")
try:
    bcrypt.checkpw(b"x", b"not-a-bcrypt-hash")
except ValueError:
    print("checkpw rejected a malformed stored hash")
else:
    raise AssertionError("checkpw accepted a malformed stored hash")

# bcrypt only uses the first 72 bytes of a password. bcrypt 5.0 and later reject longer
# input with ValueError; earlier releases truncate it silently.
long_pw = b"a" * 72 + b"-tail"
try:
    long_hash = bcrypt.hashpw(long_pw, salt)
except ValueError as exc:
    assert "72" in str(exc), exc
    print("passwords over 72 bytes are rejected")
else:
    assert bcrypt.checkpw(long_pw[:72], long_hash), "the hash does not match the first 72 bytes"
    print("passwords over 72 bytes are truncated to 72 bytes")